# AMEX Enterprise Credit Risk Platform
## Notebook 35 -- Early Payment Default: Modeling (AUC-Retention Curve)
### Phase 2 . Problem Statement 5: Early Payment Default Detection

CRISP-DM stage: **Modeling**. Sprint 1, Notebook 2 of 4 for this problem. Depends on Problem 1 Notebooks 01-05 (reads `project_config.json`, `notebook_02_summary.json`, `notebook_05_summary.json`, and the raw Kaggle CSVs) and this problem's own Notebook 34 (`notebook_34_summary.json` -> `early_default_policy.json`) -- no dependency on Problem 3 or Problem 4.

**What this notebook does (real, computed on your machine when you run it):**
- For EACH candidate early window from Notebook 34 (`EARLY_WINDOW_CANDIDATES`, e.g. `[3, 6, 9, 12]`), streams the raw Kaggle `train_data.csv` and restricts every customer to their first K chronologically-earliest statements (real `S_2` order)
- Re-engineers the SAME feature set Notebooks 02 (base mean/std/min/max/last, statement_count, tenure_days) and 04 (trend_slope/trend_delta, ratio features, interaction terms) already established -- just scoped to that truncated window, computed via a single consolidated raw-data pass per K
- Trains Problem 1's real champion ARCHITECTURE (same algorithm + hyperparameters Notebook 05 selected) on each K's restricted feature set, reusing the EXACT same train/validation customer split Notebook 02 established, so every K's holdout AUC is measured on the identical population Notebook 05's full-history AUC was measured on
- Reports an AUC-retention curve across window lengths against Notebook 05's real full-history holdout AUC, and checks each K against the KPI target Notebook 34 set (>=80% AUC retention)

**Scope note (explicit ASSUMPTION):** this notebook evaluates only the champion architecture at each K, not a full re-run of Notebook 05's multi-model tournament -- see Section 7's printed SCOPE explanation for why.

**What this notebook does NOT do:** no statistical validation or deployment packaging yet -- that's Notebook 36. This notebook only answers "how much predictive power survives at each early window length".

Zero-fabrication: every number this notebook prints is computed live from your real Kaggle data on this run, reusing Notebook 34's real, measured `EARLY_WINDOW_CANDIDATES` policy (not a re-guessed value) and Notebook 05's real, measured champion metrics as the comparison baseline.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05)
#             AND PROBLEM 5'S OWN NOTEBOOK 34 (EARLY-WINDOW POLICY)
# =============================================================================
import os
import sys
import gc
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05 and 34")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB34_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_34_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB34_SUMMARY_PATH, "run 34_early_payment_default_business_understanding.ipynb first (this "
                         "notebook consumes its EARLY_WINDOW_CANDIDATES policy, not a guess)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB34_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB34_SUMMARY = json.load(f)

POLICY_PATH = Path(NB34_SUMMARY["policy_path"])
if not POLICY_PATH.exists():
    raise FileNotFoundError(
        f"{POLICY_PATH} not found (path recorded in notebook_34_summary.json).\n"
        "Fix: re-run 34_early_payment_default_business_understanding.ipynb."
    )
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    EARLY_DEFAULT_POLICY = json.load(f)

EARLY_WINDOW_CANDIDATES = EARLY_DEFAULT_POLICY["early_window_candidates"]
EARLY_WINDOW_COVERAGE = EARLY_DEFAULT_POLICY["early_window_coverage_by_k"]
STATEMENT_COUNT_STATS = EARLY_DEFAULT_POLICY["statement_count_stats"]
EPD_KPI_TARGETS = EARLY_DEFAULT_POLICY["kpi_targets"]

if not EARLY_WINDOW_CANDIDATES:
    raise RuntimeError(
        "notebook_34_summary.json / early_default_policy.json has an empty "
        "early_window_candidates list -- re-run Notebook 34 (this looks like an "
        "older, pre-fix policy file)."
    )

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

if "early_payment_default_modeling" in PILLAR_DIRS:
    EPD_MODELING_DIR = PILLAR_DIRS["early_payment_default_modeling"]
else:
    EPD_MODELING_DIR = (
        PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning"
        / "05_Problem5_Early_Payment_Default_Detection" / "modeling"
    )
    print(
        "NOTE: 'early_payment_default_modeling' not found in project_config.json's "
        "pillar_dirs -- using the standard folder-convention fallback:\n"
        f"      {EPD_MODELING_DIR}"
    )
EPD_MODELING_DIR.mkdir(parents=True, exist_ok=True)

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")
FULL_HISTORY_AMEX_METRIC = CHAMPION_METRICS.get("holdout_amex_metric")

print(f"Loaded config from        : {CONFIG_PATH}")
print(f"Loaded early-window policy: {POLICY_PATH}")
print(f"RANDOM_SEED                : {RANDOM_SEED}")
print(f"WARP_THREAD_COUNT          : {WARP_THREAD_COUNT}")
print(f"Champion architecture (Problem 1, measured) : {CHAMPION_NAME}")
print(f"Champion holdout AUC (measured, reference)  : {FULL_HISTORY_AUC}")
print(f"Champion holdout AMEX metric (measured, ref): {FULL_HISTORY_AMEX_METRIC}")
print(f"EARLY_WINDOW_CANDIDATES (from Notebook 34)  : {EARLY_WINDOW_CANDIDATES}")
for _k in EARLY_WINDOW_CANDIDATES:
    print(f"  K={_k:>2}: {EARLY_WINDOW_COVERAGE[str(_k)] if str(_k) in EARLY_WINDOW_COVERAGE else EARLY_WINDOW_COVERAGE.get(_k):.1f}% coverage (from Notebook 34)")
print(f"Modeling artifacts will be written under: {EPD_MODELING_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    from sklearn.metrics import roc_auc_score
except ImportError:
    missing.append("scikit-learn")
try:
    from xgboost import XGBClassifier
except ImportError:
    missing.append("xgboost")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}\n"
        "Note: xgboost is required here (not optional) because this notebook "
        "specifically tests Problem 1's real champion architecture (xgboost) "
        "at restricted early windows -- see Section 1's printed champion name."
    )


def _rss_gb() -> float:
    """Current process resident memory, in GB."""
    return psutil.Process().memory_info().rss / 1e9


def amex_metric_numpy(y_true: "np.ndarray", y_pred: "np.ndarray") -> float:
    """Official American Express - Default Prediction competition metric:
    0.5 * (Normalized Weighted Gini) + 0.5 * (Top-4% Capture Rate). Byte-for-
    byte the same implementation as Notebook 05 Section 3 -- reused here
    (not re-derived) so the metric definition can never drift between the
    full-history champion and this notebook's restricted-window models."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    def top_four_percent_captured(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        cum_weight = np.cumsum(weight)
        cutoff = 0.04 * weight.sum()
        mask = cum_weight <= cutoff
        total_pos = yt_sorted.sum()
        if total_pos == 0:
            return 0.0
        return float(yt_sorted[mask].sum() / total_pos)

    def weighted_gini(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        random_cum = np.cumsum(weight / weight.sum())
        total_pos_weighted = (yt_sorted * weight).sum()
        if total_pos_weighted == 0:
            return 0.0
        cum_pos_found = np.cumsum(yt_sorted * weight)
        lorentz = cum_pos_found / total_pos_weighted
        return float(((lorentz - random_cum) * weight).sum())

    g_actual = weighted_gini(y_true, y_pred)
    g_perfect = weighted_gini(y_true, y_true)
    normalized_gini = g_actual / g_perfect if g_perfect != 0 else 0.0
    top4 = top_four_percent_captured(y_true, y_pred)
    return 0.5 * (normalized_gini + top4)


def top_four_percent_capture_only(y_true, y_pred) -> float:
    """Standalone top-4% capture rate -- same implementation as Notebook 05."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    order = np.argsort(-y_pred, kind="mergesort")
    yt_sorted = y_true[order]
    weight = np.where(yt_sorted == 0, 20.0, 1.0)
    cum_weight = np.cumsum(weight)
    cutoff = 0.04 * weight.sum()
    mask = cum_weight <= cutoff
    total_pos = yt_sorted.sum()
    if total_pos == 0:
        return 0.0
    return float(yt_sorted[mask].sum() / total_pos)


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("amex_metric_numpy() / top_four_percent_capture_only() defined (same as Notebook 05 Section 3).")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS (RAW CSVs + NOTEBOOK 02's SPLIT FILES)
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

# --- Raw train_data.csv / train_labels.csv: same directory-resolution
#     pattern Notebook 34 Section 4 established (train_full_features.parquet
#     is already aggregated to one row per customer -- confirmed on a real
#     run -- so genuine per-statement data only exists in the raw Kaggle
#     CSVs, which this platform's data/README.md documents as not
#     redistributed inside the project folder). ---
_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break

if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n"
        + "\n".join(f"  - {c}" for c in _raw_candidates)
        + "\n\nFix: tell me the real path to your raw train_data.csv."
    )

RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(
        f"{RAW_TRAIN_LABELS_PATH} not found (expected alongside {RAW_TRAIN_DATA_PATH.name})."
    )

print(f"Raw train_data.csv   : {RAW_TRAIN_DATA_PATH} ({RAW_TRAIN_DATA_PATH.stat().st_size / 1e9:.2f} GB)")
print(f"Raw train_labels.csv : {RAW_TRAIN_LABELS_PATH} ({RAW_TRAIN_LABELS_PATH.stat().st_size / 1e6:.2f} MB)")


# --- train_split.csv / test_split.csv: these customer_ID lists come out of
#     notebook_02_summary.json's "output_files" dict, which was written
#     BEFORE the Phase 1 folder reorg and still holds the pre-reorg absolute
#     paths on this project (same root cause documented for Notebooks 27/28
#     and Notebook 34's Section 4 -- see the main project's bugfix log).
#     Reusing the established 3-candidate resolver: current Phase/Problem
#     structure first, legacy root-level structure second, exactly what the
#     summary JSON says third -- each candidate only counts if it exists AND
#     is non-trivially sized, guarding against a stub/corrupt file.
#
#     CORRECTION (found on a real run): this notebook originally trusted
#     PILLAR_DIRS[<pillar>] (from the CURRENT project_config.json) as
#     automatically reflecting the post-reorg folder structure. On this real
#     project that assumption was WRONG for pillars that existed before the
#     reorg -- project_config.json's pillar_dirs entries for "data_engineering"
#     AND "data_validation" both still hold their pre-reorg root-level paths
#     (project_config.json itself was never regenerated after the Phase 1
#     folder move). Only brand-new pillars a notebook creates fresh at run
#     time (e.g. Notebook 34's EPD_POLICY_DIR) are guaranteed current. The fix
#     is generic, not per-file: always list the REAL KNOWN current nested
#     Phase/Problem path explicitly as the first candidate, never derive it
#     from PILLAR_DIRS alone. ---
def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + f"\n\nnotebook_02_summary.json['output_files'] keys: "
        f"{sorted(NB02_SUMMARY.get('output_files', {}).keys())}\n"
        "Fix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"train_split.csv (internal train, Notebook 02's real split) : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal holdout, Notebook 02's real split): {TEST_SPLIT_PATH}")

VALIDATION_REPORT_PATH = _resolve_pillar_file(
    "data_validation_report.json", "data_validation", "Data_Validation", min_size=100,
)
with open(VALIDATION_REPORT_PATH, "r", encoding="utf-8") as f:
    VALIDATION_REPORT = json.load(f)
print(f"data_validation_report.json                                 : {VALIDATION_REPORT_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA DETECTION & INTERACTION BASE COLUMNS
# =============================================================================
_section("SECTION 4: Live Schema Detection & Interaction Base Columns")

# Known from the official AMEX data dictionary (same documented constant
# every notebook in this platform uses -- not re-derived from the header,
# since dtype isn't recoverable from a header row alone).
CATEGORICAL = ["B_30", "B_38", "D_63", "D_64", "D_66", "D_68",
               "D_114", "D_116", "D_117", "D_120", "D_126"]

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
feature_columns = [c for c in _train_header if c not in ("customer_ID", "S_2")]
categorical_cols = [c for c in feature_columns if c in CATEGORICAL]
numeric_cols = [c for c in feature_columns if c not in CATEGORICAL]
print(f"Live-read header from {RAW_TRAIN_DATA_PATH.name}.")
print(f"Numeric columns    : {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)} -> {categorical_cols}")

# --- Interaction terms reuse the EXACT same base columns Notebook 03 found,
#     live, to have the highest |correlation with target| in the FULL-HISTORY
#     data -- kept IDENTICAL across every candidate K (an explicit
#     ASSUMPTION) so the AUC-retention comparison across window lengths isn't
#     confounded by also re-selecting different interaction columns at each
#     K; only the restricted TIME WINDOW changes between runs, nothing else
#     about which features are engineered. ---
_top_corr_cols = list(VALIDATION_REPORT["top_5_correlated_with_target"].keys())
print(f"\nInteraction base columns (top |corr(feature, target)| from Notebook 03, held fixed across all K): "
      f"{_top_corr_cols}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: RESTRICTED-WINDOW FEATURE ENGINEERING -- REUSABLE FUNCTIONS
# =============================================================================
_section("SECTION 5: Restricted-Window Feature Engineering -- Reusable Functions")


def build_early_window_store(csv_path: Path, num_cols: list, cat_cols: list, k: int) -> "pl.DataFrame":
    """Streams csv_path and returns one aggregated row per customer_ID,
    restricted to each customer's first K chronologically-earliest
    statements (by real S_2 date order -- customers with fewer than K real
    statements simply use all of them, same semantics as Notebook 34's
    coverage percentages).

    Computes the SAME per-customer statistics Notebook 02 (mean/std/min/max/
    last per numeric column, last/nunique per categorical column,
    statement_count, tenure_days) and Notebook 04 (trend_slope/trend_delta
    per numeric column, via the cov/var slope identity) compute over full
    history -- just scoped to the truncated window. This consolidates what
    Notebooks 02 and 04 do as two separate raw-data passes into a single
    pass here, since both need the same per-customer row-position index
    (_t_idx) already required to perform the K-truncation -- an efficiency
    consolidation, not a methodology change; every individual statistic is
    defined identically to Notebooks 02/04, including the same inf-token
    cleaning and the same var(t)-masked-to-null-where-y-is-null correctness
    fix Notebook 04 established for the trend slope.
    """
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in cat_cols:
        schema_overrides[c] = pl.Utf8
    for c in num_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in num_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
        .filter(pl.col("_t_idx") < k)
    )

    agg_exprs = []
    for c in num_cols:
        agg_exprs += [
            pl.col(c).mean().alias(f"{c}_mean"),
            pl.col(c).std().alias(f"{c}_std"),
            pl.col(c).min().alias(f"{c}_min"),
            pl.col(c).max().alias(f"{c}_max"),
            pl.col(c).last().alias(f"{c}_last"),
            pl.col(c).first().alias(f"_first_{c}"),
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None)
              .var().alias(f"_var_t_{c}"),
        ]
    for c in cat_cols:
        agg_exprs += [
            pl.col(c).last().alias(f"{c}_last"),
            pl.col(c).drop_nulls().n_unique().alias(f"{c}_nunique"),
        ]
    agg_exprs += [
        pl.len().alias("statement_count"),
        (pl.col("S_2").max() - pl.col("S_2").min()).dt.total_days().alias("tenure_days"),
    ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)

    _trend_exprs = []
    for c in num_cols:
        _trend_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}"))
            .otherwise(None)
            .alias(f"{c}_trend_slope")
        )
        _trend_exprs.append((pl.col(f"{c}_last") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta"))

    _keep_cols = ["customer_ID", "statement_count", "tenure_days"]
    for c in num_cols:
        _keep_cols += [f"{c}_mean", f"{c}_std", f"{c}_min", f"{c}_max", f"{c}_last"]
    for c in cat_cols:
        _keep_cols += [f"{c}_last", f"{c}_nunique"]
    _keep_cols += [f"{c}_trend_slope" for c in num_cols] + [f"{c}_trend_delta" for c in num_cols]

    result = grouped.with_columns(_trend_exprs).select(_keep_cols).sort("customer_ID")
    return result.collect(engine="streaming")


def build_ratio_features(df: "pl.DataFrame", num_cols: list) -> "pl.DataFrame":
    """Identical formulas to Notebook 04 Section 5 -- purely derived from the
    mean/std/min/max/last columns build_early_window_store() already
    computed, no additional raw-data scan needed."""
    exprs, new_col_names = [], []
    for c in num_cols:
        _mean, _std, _min, _max, _last = f"{c}_mean", f"{c}_std", f"{c}_min", f"{c}_max", f"{c}_last"
        if not all(col in df.columns for col in (_mean, _std, _min, _max, _last)):
            continue
        _ratio_name, _range_name, _cov_name = (
            f"{c}_ratio_last_to_mean", f"{c}_range", f"{c}_coeff_of_variation",
        )
        exprs.append(
            pl.when((pl.col(_mean).is_not_null()) & (pl.col(_mean) != 0))
            .then(pl.col(_last) / pl.col(_mean)).otherwise(None).alias(_ratio_name)
        )
        exprs.append((pl.col(_max) - pl.col(_min)).alias(_range_name))
        exprs.append(
            pl.when((pl.col(_mean).is_not_null()) & (pl.col(_mean) != 0))
            .then(pl.col(_std) / pl.col(_mean)).otherwise(None).alias(_cov_name)
        )
        new_col_names += [_ratio_name, _range_name, _cov_name]
    return df.with_columns(exprs).select(["customer_ID"] + new_col_names)


def build_interaction_features(df: "pl.DataFrame", top_corr_cols: list) -> "pl.DataFrame":
    """Identical construction to Notebook 04 Section 6, applied to whichever
    top-correlated columns are actually present in this K's base frame."""
    _cols = [c for c in top_corr_cols if c in df.columns]
    exprs, pairs = [], []
    for i in range(len(_cols)):
        for j in range(i + 1, len(_cols)):
            c1, c2 = _cols[i], _cols[j]
            _name = f"interaction_{c1}_x_{c2}"
            exprs.append((pl.col(c1) * pl.col(c2)).alias(_name))
            pairs.append(_name)
    if not pairs:
        return df.select(["customer_ID"])
    return df.select(["customer_ID"] + _cols).with_columns(exprs).select(["customer_ID"] + pairs)


print("build_early_window_store(), build_ratio_features(), build_interaction_features() defined.")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: LOAD LABELS & TRAIN/VALIDATION SPLIT MEMBERSHIP
# =============================================================================
_section("SECTION 6: Load Labels & Train/Validation Split Membership")

labels_df = pl.read_csv(str(RAW_TRAIN_LABELS_PATH), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
print(f"Live-read {RAW_TRAIN_LABELS_PATH.name}: {labels_df.shape[0]:,} labeled customers")

# --- Reuses the EXACT same train/validation split membership Notebook 02
#     established (customer_ID lists), so every candidate K's holdout AUC is
#     measured on the IDENTICAL population Notebook 05's champion AUC was
#     measured on -- a genuine apples-to-apples comparison, not a re-split. ---
train_ids_set = set(pl.read_csv(str(TRAIN_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
val_ids_set = set(pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
print(f"Train-split customers (from Notebook 02, reused)    : {len(train_ids_set):,}")
print(f"Validation-split customers (from Notebook 02, reused): {len(val_ids_set):,}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: BUILD RESTRICTED FEATURE SETS & TRAIN/EVALUATE THE CHAMPION
#            ARCHITECTURE -- FOR EACH CANDIDATE K
# =============================================================================
_section("SECTION 7: Build & Evaluate the Champion Architecture at Each Candidate K")

print(
    "SCOPE (ASSUMPTION, stated plainly): this notebook evaluates only Problem 1's "
    f"real champion ARCHITECTURE ({CHAMPION_NAME}, same hyperparameters Notebook 05 "
    "used for it) at each candidate early window -- it does not re-run Notebook 05's "
    "full multi-model tournament (5-fold CV across 5-7 algorithms) at every K, since "
    "that would be a ~4x-7x larger compute cost to answer a question this notebook "
    "isn't asking. The question here is whether RESTRICTING THE FEATURES (fewer "
    "early statements) preserves predictive power, not which algorithm is best -- so "
    "holding the algorithm fixed at the already-established champion keeps the "
    "comparison focused on that one variable."
)

categorical_encode_cols = [f"{c}_last" for c in CATEGORICAL if c in categorical_cols]

EARLY_WINDOW_MODELING_RESULTS = {}

for _k in EARLY_WINDOW_CANDIDATES:
    _section(f"  K={_k}: Building Restricted Feature Set")
    gc.collect()
    _t0 = time.time()

    _base = build_early_window_store(RAW_TRAIN_DATA_PATH, numeric_cols, categorical_cols, _k)
    _ratios = build_ratio_features(_base, numeric_cols)
    _interactions = build_interaction_features(_base, _top_corr_cols)

    engineered = (
        _base
        .join(_ratios, on="customer_ID", how="left")
        .join(_interactions, on="customer_ID", how="left")
        .join(labels_df, on="customer_ID", how="inner")
    )
    if engineered.shape[0] != _base.shape[0]:
        raise RuntimeError(
            f"K={_k}: row count changed after joining ratios/interactions/labels "
            f"({_base.shape[0]:,} -> {engineered.shape[0]:,}). Fix: investigate a "
            "many-to-one join or customer_ID mismatch before proceeding."
        )
    _build_seconds = time.time() - _t0
    print(f"K={_k}: built {engineered.shape[0]:,} customers x {engineered.shape[1]} columns "
          f"in {_build_seconds:.1f}s. Process RSS: {_rss_gb():.2f} GB")
    del _base, _ratios, _interactions
    gc.collect()

    train_df = engineered.filter(pl.col("customer_ID").is_in(train_ids_set))
    holdout_df = engineered.filter(pl.col("customer_ID").is_in(val_ids_set))
    del engineered
    gc.collect()

    non_feature_cols = {"customer_ID", "target"}
    numeric_feature_cols = [c for c in train_df.columns if c not in non_feature_cols and c not in categorical_encode_cols]
    all_feature_cols = numeric_feature_cols + categorical_encode_cols

    # --- Preprocessing: identical steps to Notebook 05 Section 5 (inf/nan ->
    #     null, label-encode categoricals fit on TRAIN split only, median-
    #     impute fit on TRAIN split only) -- just scoped to this K's frames. ---
    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
        for c in numeric_feature_cols
    ]
    train_df = train_df.with_columns(_inf_clean_exprs)
    holdout_df = holdout_df.with_columns(_inf_clean_exprs)

    for c in categorical_encode_cols:
        train_df = train_df.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
        holdout_df = holdout_df.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
        _cats = sorted(train_df.get_column(c).unique().to_list())
        _mapping = {cat: i for i, cat in enumerate(_cats)}
        train_df = train_df.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
        holdout_df = holdout_df.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))

    _feature_medians = train_df.select(
        [pl.col(c).median().fill_null(0.0).alias(c) for c in numeric_feature_cols]
    ).to_dicts()[0]
    _impute_exprs = [pl.col(c).fill_null(_feature_medians[c]) for c in numeric_feature_cols]
    train_df = train_df.with_columns(_impute_exprs)
    holdout_df = holdout_df.with_columns(_impute_exprs)

    X_train = train_df.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
    y_train = train_df.get_column("target").to_numpy().astype(np.int64, copy=False)
    X_holdout = holdout_df.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
    y_holdout = holdout_df.get_column("target").to_numpy().astype(np.int64, copy=False)
    del train_df, holdout_df
    gc.collect()

    print(f"K={_k}: X_train {X_train.shape}, X_holdout {X_holdout.shape}, "
          f"{len(all_feature_cols)} features ({len(categorical_encode_cols)} categorical)")

    # --- Train the champion architecture, same hyperparameters Notebook 05
    #     Section 6 used for it (n_estimators=400, max_depth=6,
    #     learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    #     tree_method="hist"), on THIS K's restricted feature matrix only. ---
    _t0 = time.time()
    model = XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
        eval_metric="auc", verbosity=0,
    )
    model.fit(X_train, y_train)
    _train_seconds = time.time() - _t0

    proba = model.predict_proba(X_holdout)[:, 1]
    holdout_auc = float(roc_auc_score(y_holdout, proba))
    holdout_amex = float(amex_metric_numpy(y_holdout, proba))
    holdout_top4 = float(top_four_percent_capture_only(y_holdout, proba))
    auc_retention_pct = float(holdout_auc / FULL_HISTORY_AUC * 100.0) if FULL_HISTORY_AUC else None
    meets_kpi = (
        (holdout_auc / FULL_HISTORY_AUC) >= EPD_KPI_TARGETS["min_auc_retention_vs_full_history"]
        if FULL_HISTORY_AUC else False
    )

    EARLY_WINDOW_MODELING_RESULTS[_k] = {
        "k": _k,
        "coverage_pct": EARLY_WINDOW_COVERAGE.get(str(_k), EARLY_WINDOW_COVERAGE.get(_k)),
        "train_customers": int(X_train.shape[0]),
        "holdout_customers": int(X_holdout.shape[0]),
        "feature_count": len(all_feature_cols),
        "holdout_auc": holdout_auc,
        "holdout_amex_metric": holdout_amex,
        "holdout_top4pct_capture": holdout_top4,
        "auc_retention_pct_of_full_history": auc_retention_pct,
        "meets_kpi_target": bool(meets_kpi),
        "train_seconds": round(_train_seconds, 1),
    }
    print(f"K={_k:>2}  Holdout AUC {holdout_auc:.4f}  (retains {auc_retention_pct:.1f}% of full-history AUC "
          f"{FULL_HISTORY_AUC:.4f})  Holdout AMEX {holdout_amex:.4f}  "
          f"KPI (>= {EPD_KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} retention): "
          f"{'MET' if meets_kpi else 'NOT MET'}  (train {_train_seconds:.1f}s)")

    del X_train, y_train, X_holdout, y_holdout, model, proba
    gc.collect()

print(f"\nProcess RSS after all {len(EARLY_WINDOW_CANDIDATES)} candidate windows: {_rss_gb():.2f} GB")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: AUC-RETENTION CURVE -- SUMMARY & HONEST INTERPRETATION
# =============================================================================
_section("SECTION 8: AUC-Retention Curve -- Summary & Interpretation")

print(f"{'K':>4} {'Coverage%':>10} {'Holdout AUC':>12} {'Retention%':>11} {'Holdout AMEX':>13} {'KPI':>8}")
for _k in EARLY_WINDOW_CANDIDATES:
    _r = EARLY_WINDOW_MODELING_RESULTS[_k]
    print(f"{_k:>4} {_r['coverage_pct']:>9.1f}% {_r['holdout_auc']:>12.4f} "
          f"{_r['auc_retention_pct_of_full_history']:>10.1f}% {_r['holdout_amex_metric']:>13.4f} "
          f"{'MET' if _r['meets_kpi_target'] else 'not met':>8}")

_ks_meeting_kpi = [k for k in EARLY_WINDOW_CANDIDATES if EARLY_WINDOW_MODELING_RESULTS[k]["meets_kpi_target"]]
if _ks_meeting_kpi:
    _earliest_viable_k = min(_ks_meeting_kpi)
    print(
        f"\nEarliest candidate window meeting the {EPD_KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} "
        f"AUC-retention KPI target: K={_earliest_viable_k} "
        f"({EARLY_WINDOW_MODELING_RESULTS[_earliest_viable_k]['coverage_pct']:.1f}% of customers have that "
        f"many statements). This is the shortest early-observation window this run's real data supports for "
        "operationally useful early risk flagging, given the KPI target set in Notebook 34."
    )
else:
    print(
        f"\nHONEST FINDING: none of the tested candidate windows ({EARLY_WINDOW_CANDIDATES}) retained "
        f"{EPD_KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} of the full-history champion AUC on "
        "this real run. This is reported plainly rather than obscured -- the KPI target itself, or the set "
        "of candidate windows tested, may need revisiting; that is a business decision, not something this "
        "notebook should paper over."
    )
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WRITE MODELING RESULTS ARTIFACT
# =============================================================================
_section("SECTION 9: Write Modeling Results Artifact")

EARLY_WINDOW_MODELING_ARTIFACT = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 5 -- Early Payment Default Detection",
    "champion_architecture_used": CHAMPION_NAME,
    "champion_architecture_scope_note": (
        "Only the champion architecture is evaluated at each K (see Section 7's "
        "printed SCOPE note) -- this is not a re-run of Notebook 05's full "
        "multi-model tournament."
    ),
    "full_history_reference_auc": FULL_HISTORY_AUC,
    "full_history_reference_amex_metric": FULL_HISTORY_AMEX_METRIC,
    "results_by_k": {str(k): v for k, v in EARLY_WINDOW_MODELING_RESULTS.items()},
    "ks_meeting_kpi_target": _ks_meeting_kpi,
    "kpi_targets": EPD_KPI_TARGETS,
    "random_seed": RANDOM_SEED,
}

MODELING_RESULTS_PATH = EPD_MODELING_DIR / "early_window_modeling_results.json"
with open(MODELING_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(EARLY_WINDOW_MODELING_ARTIFACT, f, indent=2)
print(f"Wrote: {MODELING_RESULTS_PATH}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 10: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Modeling results file was written", MODELING_RESULTS_PATH.exists())
_all_checks_passed &= _check(
    "A result was computed for every candidate K",
    set(EARLY_WINDOW_MODELING_RESULTS.keys()) == set(EARLY_WINDOW_CANDIDATES),
)
_all_checks_passed &= _check(
    "Every holdout AUC is a real value in (0.5, 1.0] (better than random, at most perfect)",
    all(0.5 < r["holdout_auc"] <= 1.0 for r in EARLY_WINDOW_MODELING_RESULTS.values()),
)
_all_checks_passed &= _check(
    "Every holdout AMEX metric is a real value in (0.0, 1.0]",
    all(0.0 < r["holdout_amex_metric"] <= 1.0 for r in EARLY_WINDOW_MODELING_RESULTS.values()),
)
_all_checks_passed &= _check(
    "auc_retention_pct is correctly derived (holdout_auc / full_history_auc)",
    all(
        abs(r["auc_retention_pct_of_full_history"] / 100.0 - (r["holdout_auc"] / FULL_HISTORY_AUC)) < 1e-6
        for r in EARLY_WINDOW_MODELING_RESULTS.values()
    ),
)
_all_checks_passed &= _check(
    "Reused Problem 1's real champion AUC (not fabricated)",
    FULL_HISTORY_AUC == CHAMPION_METRICS.get("holdout_auc"),
)
_all_checks_passed &= _check(
    "Every K's train+holdout customer count matches the reused Notebook 02 split sizes",
    all(
        r["train_customers"] == len(train_ids_set) and r["holdout_customers"] == len(val_ids_set)
        for r in EARLY_WINDOW_MODELING_RESULTS.values()
    ),
)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 10 complete -- all checks passed.")


# =============================================================================
# SECTION 11: WRITE NOTEBOOK 35 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 11: Write Notebook 35 Summary Artifact")

NB35_SUMMARY = {
    "notebook": "35_early_payment_default_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "champion_architecture_used": CHAMPION_NAME,
    "results_by_k": {str(k): v for k, v in EARLY_WINDOW_MODELING_RESULTS.items()},
    "ks_meeting_kpi_target": _ks_meeting_kpi,
    "modeling_results_path": str(MODELING_RESULTS_PATH),
    "random_seed": RANDOM_SEED,
}
NB35_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_35_summary.json"
with open(NB35_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB35_SUMMARY, f, indent=2)
print(f"Wrote: {NB35_SUMMARY_PATH}")

_section("NOTEBOOK 35 COMPLETE")
for _k in EARLY_WINDOW_CANDIDATES:
    _r = EARLY_WINDOW_MODELING_RESULTS[_k]
    print(f"  K={_k:>2}: AUC {_r['holdout_auc']:.4f} ({_r['auc_retention_pct_of_full_history']:.1f}% retention) "
          f"-- KPI {'MET' if _r['meets_kpi_target'] else 'not met'}")
if _ks_meeting_kpi:
    print(f"\nEarliest viable window: K={min(_ks_meeting_kpi)}")
else:
    print("\nNo candidate window met the KPI target on this real run (see Section 8).")
print(
    "\nNext: 36_early_payment_default_validation_deployment.ipynb -- statistical "
    "validation and deployment packaging for the earliest viable window found above "
    "(or an honest 'not yet viable' conclusion if none met the KPI)."
)
